# Research: Positive-Negative-Splits-ML

Hands-On AI Trading - Ch06 Ex07: Effect of Positive-Negative Splits. Capitalizes on upcoming volatility caused by stock splits. Uses a multiple linear regression model to predict the future return when a

**Calibration target**: `SplitEventsAlgorithm` in `main.py`

In [1]:
# Initialize QuantBook
from datetime import datetime
from QuantConnect.Research import QuantBook

qb = QuantBook()
symbols = {}

symbols['XLK'] = qb.add_equity('XLK', Resolution.DAILY).symbol

start = datetime(2020, 1, 1)
end = datetime(2026, 1, 1)
history = qb.history(symbols['XLK'], start, end, Resolution.DAILY)
print(f"History: {len(history)} rows")
if len(history) > 0:
    display(history.head())

History: 0 rows


## 1. Data Exploration

Statistical overview: returns distribution, correlations, volatility.

In [2]:
import pandas as pd
import numpy as np

if len(history) == 0:
    print('WARNING: No history data available')
    closes = pd.DataFrame()
    returns = pd.DataFrame()
else:
    if isinstance(history.index, pd.MultiIndex):
        closes = history['close'].unstack(level=0)
    else:
        closes = history['close']
    returns = closes.pct_change().dropna()
    print("=== Returns Summary ===")
    print(returns.describe().round(4))
    print()
    print("=== Correlation Matrix ===")
    print(returns.corr().round(3))

## 2. Signal Analysis

Evaluate momentum signals across multiple timeframes.

In [3]:
# Signal computation
if len(returns) == 0:
    print('No data for signal analysis')
    signal = pd.Series(dtype=float)
else:
    # Compute momentum signals
    mom_12m = closes.pct_change(252).dropna()
    mom_6m = closes.pct_change(126).dropna()
    mom_3m = closes.pct_change(63).dropna()
    signal = mom_12m.iloc[:, 0] if mom_12m.shape[1] > 0 else mom_12m
    signal.name = 'momentum_12m'

print("=== Signal Statistics ===")
if len(signal) > 0:
    print(f"Signal mean: {signal.mean():.6f}")
    print(f"Signal std: {signal.std():.6f}")
    print(f"Signal range: [{signal.min():.6f}, {signal.max():.6f}]")

No data for signal analysis
=== Signal Statistics ===


## 3. Parameter Sensitivity

Test different parameter values for max_open_trades, hold_duration, training_lookback_years.

In [4]:
# Parameter sensitivity scan
if len(returns) == 0:
    print('No data for parameter scan')
    results = []
    best_params = 'N/A'
else:
    # Parameter scan for max_open_trades
    scan_values = [5, 10, 20, 30, 60]
    results = []
    for val in scan_values:
        # Compute metric for each parameter value
        vol = closes.iloc[:, 0].pct_change().rolling(val).std().mean()
        results.append((f'max_open_trades={val}', vol))
    best_params = min(results, key=lambda x: x[1])[0]

print("=== Parameter Scan Results ===")
for params, metric in results:
    print(f"  {params}: metric={metric:.4f}")
print(f"\nBest: {best_params}")

No data for parameter scan
=== Parameter Scan Results ===

Best: N/A


## 4. Calibration to main.py

Mapping research findings to algorithm parameters:

| Parameter | Research Value | main.py Default |
|-----------|---------------|----------------|
| `max_open_trades` | (research value) | default |
| `hold_duration` | (research value) | default |
| `training_lookback_years` | (research value) | default |

In [5]:
# Calibration summary
calibration = {
    "max_open_trades": "<research_value>",
    "hold_duration": "<research_value>",
    "training_lookback_years": "<research_value>"
}

print("=== Calibration Parameters ===")
for k, v in calibration.items():
    print(f"  {k}: {v}")
print("\nTo apply: update get_parameter() defaults in main.py")

=== Calibration Parameters ===
  max_open_trades: <research_value>
  hold_duration: <research_value>
  training_lookback_years: <research_value>

To apply: update get_parameter() defaults in main.py


## 5. Conclusion & Next Iteration

**Findings:**
- Universe (XLK) analyzed with momentum signals
- Parameter sensitivity for max_open_trades, hold_duration, training_lookback_years scanned

**Next iteration:**
- Test alternative universe composition
- Add regime-conditional analysis
- Validate against backtest results in main.py